* Distribution shift happens when the data a model sees after deployment differs from the data it learned from.
* This notebook explains the main shift types and why they matter for classification systems.

In [1]:
import math
import random
import numpy as np
import torch

# 4.7.1 Types of Distribution Shift

## 1. Intuition

*   A distribution describes how data values occur.
> *   Distribution shift means the training distribution and deployment distribution are different.
>> * e.g. 4K image used for training; blurry iP4 image used during deployment.
> *   Covariate shift means input features change while the label rule is assumed stable.
>> * e.g. self-driving car trained during summer deployed in winter.
> *   Label shift means class frequencies change.
>> * e.g. a fraud detector trained on 1% frauds faces 10% frauds during deployment.
> *   Concept shift means the relationship between inputs and labels changes.
>> * e.g. spam patterns evovle from ("Win a free iPhone!") to Nigerian prince.

    | Type                   | What changes?                       | Example                                 |
    | ---------------------- | ----------------------------------- | --------------------------------------- |
    | **Distribution**       | How data is spread                  | 50% cats, 50% dogs                      |
    | **Distribution shift** | Training and real-world data differ | Daytime training, nighttime deployment  |
    | **Covariate shift**    | Inputs (**X**) change               | Snowy road images instead of sunny ones |
    | **Label shift**        | Class frequencies (**Y**) change    | Disease becomes much more common        |
    | **Concept shift**      | Input-to-label relationship changes | Fraud patterns evolve                   |


## 2. Why this exists

*   Models learn patterns from training data.
*   If the future data follows different patterns, training performance may not predict deployment performance.

## 3. Examples

*   Represent class frequencies before and after deployment.

In [2]:
train_counts = torch.tensor([80, 20])
deploy_counts = torch.tensor([50, 50])

train_probs = train_counts / train_counts.sum() # [0.8. 0.2]
deploy_probs = deploy_counts / deploy_counts.sum() # [0.5. 0.5]
train_probs, deploy_probs

(tensor([0.8000, 0.2000]), tensor([0.5000, 0.5000]))

## 4. Step-by-step breakdown

* The training set has class (index) 0 much more often than class 1.

* The deployment setting has both classes equally often.

* The probability vectors show label frequencies.

* This is a toy example of label shift.

## 5. Connection to ML systems

*   Distribution shift (disparate training vs deployment example) is a central deployment risk.
*   A classifier validated on old data can fail when users, sensors, policies, or environments change.

## 6. Common confusion points

- Shift is about data-generating conditions, not just random noise.
> * Covariate shift affects inputs.
> * Label shift affects class proportions.
> * Concept shift affects the input-label relationship itself.

# 4.7.2 Examples of Distribution Shift

## 1. Intuition

* Distribution shift appears in practical systems when the world changes or the data collection process changes.

* Examples include new camera lighting, new user populations, seasonal demand, policy changes, or adversarial behavior.

## 2. Why this exists

*   Concrete examples make it easier to identify why a model's validation performance may not match real-world performance.

## 3. Examples

*   A simple feature mean shift.

In [3]:
train_feature = torch.tensor([0.1, 0.0, -0.1])
deploy_feature = torch.tensor([1.1, 1.0, 0.9])

train_mean = train_feature.mean() # (0.1+0.0-0.1)/3 = 0/3 = 0
deploy_mean = deploy_feature.mean() # (1.1+1.0+0.9)/3 = 3/3 = 1
deploy_mean - train_mean # 1 - 0 = 1

tensor(1.)

## 4. Step-by-step breakdown

* The training feature values are centered near 0.

* The deployment feature values are centered near 1.

* The mean difference is a simple signal that inputs changed.

* Real shift detection uses richer checks, but the basic idea is comparison.

## 5. Connection to ML systems

*   Monitoring input statistics is one way production ML systems detect that future data may no longer match training data.

## 6. Common confusion points

- A shifted feature does not always mean performance fails.
- Some shifts are harmless; others are severe.
- Shift can happen gradually or suddenly.
- The most dangerous shifts often affect labels or decision rules, not only input averages.

# 4.7.3 Correction of Distribution Shift

## 1. Intuition

* Correcting distribution shift means changing training, evaluation, or deployment procedures to reduce mismatch.

* Common responses include reweighting examples, collecting new data, retraining, domain adaptation, and monitoring.

## 2. Why this exists

*   Ignoring shift can make a model confidently wrong in the environment where it matters.

## 3. Examples

*   Toy reweighting for changed class frequencies.

In [4]:
train_probs = torch.tensor([0.8, 0.2])
deploy_probs = torch.tensor([0.5, 0.5])
weights = deploy_probs / train_probs # [0.5/0.8, 0.5 / 0.2] or [0.625, 2.5], adjusted the weights for train_probs to soothe over deploy_probs
weights

tensor([0.6250, 2.5000])

## 4. Step-by-step breakdown

* The weights compare deployment class probability to training class probability.
> * Class 0 gets downweighted because it was overrepresented in training.
> * Class 1 gets upweighted because it was underrepresented in training.

* This is a simplified example, not a complete correction method.

## 5. Connection to ML systems

*   Reweighting appears in class imbalance, label shift correction, and importance sampling.
*   Importance sampling means estimating a target distribution using weighted samples from another distribution.

## 6. Common confusion points

- Correction requires knowing or estimating the shift.
- Reweighting can increase variance when weights are large.
- Retraining may be safer when fresh labeled data is available.
- Monitoring should continue after correction.

# 4.7.4 A Taxonomy of Learning Problems

## 1. Intuition

* A taxonomy is a classification system for ideas.

* ML problems can be grouped by what feedback is available:
> * supervised learning has labels,
> * unsupervised learning has no labels,
> * self-supervised learning creates labels from data itself,
> * and reinforcement learning uses rewards from actions.

## 2. Why this exists

*   Different problem types have different assumptions about data, feedback, and shift.

## 3. Examples

*   Map learning problem types to feedback.

In [5]:
taxonomy = {
    "supervised": "input-label pairs",
    "unsupervised": "inputs without labels",
    "self_supervised": "labels derived from data", # labels derived from the data itself, like GPT-pretraining (next-token prediction)
    "reinforcement": "rewards after actions",
}

## 4. Step-by-step breakdown

* The dictionary names 4 learning settings.

* Each value describes the kind of feedback available.

* [**Classification is usually supervised learning because labels identify the correct class**].

* Other settings require different training objectives.

## 5. Connection to ML systems

*   Knowing the problem type helps choose evaluation methods and identify likely distribution-shift risks.

## 6. Common confusion points

- The same dataset can support different learning setups.
> * Supervised learning requires labels.
> * Reinforcement learning feedback depends on actions.
- A taxonomy helps reasoning, but real systems can combine categories.
> * LLM pre-training is primarily **self-supervised**, where the model learns by predicting the next token (or a similar objective) using labels generated automatically from the text itself.
> * **Supervised fine-tuning** (SFT) uses human-written examples to teach instruction following after pre-training.
> * LLM post-training typically includes supervised fine-tuning (SFT) on human demonstrations, followed by preference optimization (such as **RLHF** or **DPO**) using human preference data to improve helpfulness, instruction following, and safety.


# 4.7.5 Fairness, Accountability, and Transparency in Machine Learning

## 1. Intuition

* Fairness asks whether model behavior is unjust across people or groups.
* Accountability asks who is responsible for model decisions and harms.
* Transparency asks whether relevant people can understand or inspect the system.
* These are sociotechnical issues, meaning they involve both technical design and social context.

## 2. Why this exists

*   Classification systems can affect loans, jobs, healthcare, education, moderation, and safety.
*   Accuracy alone is not enough when errors have unequal consequences.

## 3. Examples

*   Compare accuracy across two groups.

In [9]:
group = torch.tensor([0, 0, 1, 1])
correct = torch.tensor([1, 1, 1, 0], dtype=torch.float32)
acc_group0 = correct[group == 0].mean() # [T, T, F, F] via group == 0, then boolean indexing keeps the True or [keep, keep, skip, skip] or correct[group == 0] -> [1, 1], and (1+1)/2 = 1
acc_group1 = correct[group == 1].mean() # [F, F, T, T] via group == 1, then boolean indexing keeps the True or [skip, skip, keep, keep] or correct[group == 1] -> [1, 0], and (1+0)/2 = 0.5
acc_group0, acc_group1

(tensor(1.), tensor(0.5000))

## 4. Step-by-step breakdown

* `group` stores a group ID for each example.

* `correct` stores whether each prediction was correct.

* Boolean indexing selects examples from one group.

* The 2 group accuracies reveal a performance gap in this tiny example.

## 5. Connection to ML systems

*   Responsible ML evaluation often reports performance by relevant subgroups, not only overall averages.

## 6. Common confusion points

- Fairness cannot be reduced to one universal metric.
- Group labels may be sensitive and require careful governance.
- Transparency does not automatically make a system fair.
- Accountability requires human and organizational decisions, not just code.

# 4.7.6 Summary

## 1. Intuition

*   Distribution shift and responsible ML concerns remind us that models live in environments, not just notebooks.

## 2. Why this exists

*   A classifier's usefulness depends on how data is collected, how deployment differs from training, and who is affected by errors.

## 3. Examples

*   A deployment checklist.

In [ ]:
checks = [
    "compare train and deployment data",
    "monitor performance over time",
    "evaluate important subgroups",
    "plan retraining or rollback",
]

## 4. Step-by-step breakdown

* The checklist names practical deployment concerns.

* Monitoring checks whether data and performance change.

* Subgroup evaluation checks whether averages hide harm.

* Retraining or rollback plans prepare for failure.

## 5. Connection to ML systems

*   These ideas become increasingly important as models move from experiments to real applications.

## 6. Common confusion points

- Validation performance is not the end of evaluation.
- Deployment data can change.
- Social impact is part of applied ML quality.
- Monitoring and documentation are engineering responsibilities.

# 4.7.7 Exercises

## 1. Intuition

*   These exercises practice recognizing shift and evaluating subgroup behavior.

## 2. Why this exists

*   The goal is to build habits that go beyond average accuracy.

## 3. Examples

*   Exercise 1: identify a label-frequency shift.

In [10]:
old = torch.tensor([90, 10])
new = torch.tensor([60, 40])
old_p = old / old.sum() # [0.9, 0.1]
new_p = new / new.sum() # [0.6, 0.4]
new_p, old_p

(tensor([0.6000, 0.4000]), tensor([0.9000, 0.1000]))

*   Exercise 2: compute group accuracy gap.

In [11]:
acc0 = torch.tensor(0.95)
acc1 = torch.tensor(0.75)
gap = acc0 - acc1 # 0.20
gap

tensor(0.2000)

## 4. Step-by-step breakdown

* Exercise 1 compares class proportions before and after a change.

* Exercise 2 computes a simple subgroup accuracy gap.

* Both are tiny signals that should trigger deeper investigation in real systems.

## 5. Connection to ML systems

*   Production classifiers often need monitoring reports that include these kinds of checks.

## 6. Common confusion points

- A gap is a warning sign, not a complete fairness audit.
- Shift detection needs domain context.
- Monitoring should be planned before deployment.
- Corrective action may require data, model, and process changes.